# Image Assignment: Plant Disease Classification

**Goal:** Classify plant leaf images into healthy / disease classes using **scikit-learn** as the main ML toolkit, with a **basic deep learning model** only if needed for comparison.

---

### High-level pipeline

```text
Dataset → Load & split → Preprocess → Features (sklearn)
                                      → Classical ML models
                                      → (Optional) Small CNN
                                      → Evaluate & compare
                                      → Discussion
```

## 1. Problem definition

### What we are solving
- **Input:** RGB images of plant leaves.
- **Output:** Discrete class label (e.g. `Tomato___Early_blight`, `Potato___healthy`).
- **Task type:** Multiclass image classification.

### Why this topic
- Clear real-world motivation (crop monitoring / early detection).
- Public labeled datasets exist → low data-collection cost.
- Fits teacher preference: classical ML (sklearn) first; CNN only as optional boost.

### Scope for this assignment (keep it small)
- Use a **subset** of classes (suggested: **6–8 classes**), not the full PlantVillage set.
- Prefer classes that look visually distinct enough for a fair first experiment.
- Fixed image size for all samples (e.g. 64×64 or 128×128).

### Success criteria
- Train/evaluate at least **two sklearn models** (e.g. SVM + Random Forest).
- Report accuracy + macro F1 + confusion matrix.
- Optionally compare against a **small CNN** and discuss when DL helps.

## 2. Dataset choice & setup

### Recommended source
- **PlantVillage** (or a Kaggle mirror / filtered subset).
- Typical layout after download:

```text
data/
  Tomato___Bacterial_spot/
  Tomato___Early_blight/
  Tomato___healthy/
  Potato___Early_blight/
  Potato___Late_blight/
  Potato___healthy/
  ...
```

### What to document in the report
- Dataset name + link.
- Number of selected classes and samples per class.
- Train / validation / test split ratios (e.g. 70/15/15 or 80/10/10).
- Whether split is **stratified** (recommended).
- Known bias: many PlantVillage images are lab-style (plain background) → mention as a limitation.

### Practical tips (before coding)
- Cap images per class (e.g. 300–500) if the full folder is huge.
- Keep class names consistent with folder names.
- Store a small `class_to_idx` mapping for reproducibility.

## 3. Data loading & exploration (EDA)

### Steps
1. Walk class folders and collect file paths + labels.
2. Count samples per class → bar chart of class distribution.
3. Show a few example images per class (grid of samples).
4. Check image sizes / formats; note any corrupt or unreadable files.
5. Inspect class imbalance and decide whether to undersample / oversample later.

### Questions EDA should answer
- Are classes roughly balanced?
- Do diseases look separable by color/texture to a human?
- Are there near-duplicates or very similar classes that may confuse the model?

### Deliverables for this section
- Class count table.
- Sample image montage.
- Short written note on expected hard pairs (e.g. early vs late blight).

## 4. Preprocessing

### Core transforms (required)
- Resize all images to a fixed shape (e.g. **64×64** for sklearn speed, or **128×128** if using a CNN).
- Convert to RGB if needed.
- Scale pixel values to `[0, 1]` or standardize (mean/std).

### Optional but useful
- Light augmentation **on training set only**: horizontal flip, small rotation, brightness jitter.
- Do **not** augment validation/test (evaluation must stay clean).

### Train / val / test discipline
- Split **before** fitting any scaler/PCA on features.
- Fit preprocessing statistics (scaler, PCA) on **train only**, then transform val/test.

### sklearn-friendly output shapes
- For classical models: each image becomes a **1D feature vector** (flattened pixels or handcrafted features).
- Keep original HxWxC tensors aside if you later train a small CNN.

## 5. Feature extraction (sklearn path)

Classical ML needs tabular features. Plan **at least one simple baseline** and **one stronger feature set**.

### Option A — Flattened pixels (baseline)
- Resize → flatten → optional PCA to reduce dimensionality.
- Pros: trivial to implement.
- Cons: ignores spatial structure; often weakest accuracy.

### Option B — Color histograms
- Compute per-channel histograms (RGB or HSV).
- Captures disease color cues (yellowing, brown spots).

### Option C — Texture features (recommended)
- **HOG** (Histogram of Oriented Gradients) and/or **LBP** (Local Binary Patterns).
- Better for leaf vein / lesion texture patterns.

### Option D — Combined features
- Concatenate color + texture features, then StandardScaler (+ optional PCA).

### Implementation notes (for later coding)
- Use `sklearn.decomposition.PCA`, `StandardScaler`, and feature builders from `skimage` if allowed.
- Cache extracted features to `.npz` so you do not recompute every run.
- Document feature vector length for each option.

## 6. Classical ML models (sklearn — main focus)

### Models to train (suggested set)
1. **Logistic Regression** — linear baseline, fast, interpretable coefficients (after PCA).
2. **SVM (RBF or linear)** — strong on medium-size feature vectors.
3. **Random Forest** or **Gradient Boosting** — non-linear ensemble, handles mixed features well.

### Training protocol
- Use stratified k-fold CV on the training set for hyperparameter search (`GridSearchCV` / `RandomizedSearchCV`).
- Refit best params on full training set; evaluate once on held-out test set.
- Fix `random_state` everywhere for reproducibility.

### Hyperparameters worth tuning (keep search small)
- SVM: `C`, `gamma` (if RBF), kernel.
- Random Forest: `n_estimators`, `max_depth`, `min_samples_leaf`.
- Logistic Regression: `C`, penalty if applicable.
- PCA: number of components (if used).

### Pipelines
- Prefer `sklearn.pipeline.Pipeline`: `Scaler → (PCA) → Classifier` so preprocessing never leaks.

## 7. Optional: basic deep learning model

Use this section only if classical models plateau or the assignment asks for a neural comparison.

### Model idea (keep it basic)
- Small CNN from scratch, e.g.:
  - Conv → ReLU → MaxPool (×2 or ×3 blocks)
  - Flatten / GlobalAveragePooling
  - Dense → Softmax (num_classes)
- Avoid huge transfer-learning backbones unless explicitly allowed.

### Training details to plan
- Loss: categorical / sparse categorical cross-entropy.
- Optimizer: Adam, small learning rate.
- Batch size, epochs, early stopping on validation loss/accuracy.
- Same train/val/test split as sklearn experiments for fair comparison.

### Fair comparison rules
- Same classes, same images, same test set.
- Report number of parameters and approximate training time.
- Discuss whether CNN gains justify the extra complexity.

## 8. Evaluation & metrics

### Metrics (report all of these)
- **Accuracy**
- **Precision / Recall / F1** (macro average for multiclass)
- **Confusion matrix** (normalized + raw counts)
- Optional: per-class F1 table

### Plots / tables to include
- Model comparison table (features × classifier × metrics).
- Confusion matrices for best sklearn model and (if any) CNN.
- Training curves for CNN (loss/accuracy vs epoch).

### Error analysis
- Show misclassified examples.
- List most confused class pairs and hypothesize why (similar lesions, lighting, color overlap).
- Note effect of class imbalance if present.

## 9. Results discussion

### Questions to answer in writing
1. Which feature set helped sklearn the most (pixels vs color vs HOG/LBP)?
2. Which classifier won under the same features?
3. Did the small CNN beat sklearn? By how much, and at what cost?
4. Where does the model fail, and is that acceptable for the use case?
5. How would results change on real farm photos (different backgrounds, lighting)?

### Limitations (plan to mention)
- Lab-style dataset ≠ field conditions.
- Limited class subset.
- Possible data leakage if augmentation or scaling is done incorrectly.
- Small CNN may overfit without enough regularization/data.

## 10. Conclusion & future work

### Conclusion template
- Restate task and constraints (sklearn-first).
- Summarize best method + key metric.
- One sentence on practical takeaway for plant disease screening.

### Future work ideas (pick 2–3)
- Expand to more crops/diseases.
- Stronger texture / deep features + sklearn classifier hybrid.
- Field-collected images / domain adaptation.
- Lightweight mobile deployment for on-device inference.
- Class imbalance techniques (class weights, focal loss for CNN).

## 11. Implementation checklist (for later coding)

Use this as a todo list when you start writing code:

- [ ] Download / place dataset under `data/`
- [ ] Select 6–8 classes and cap samples if needed
- [ ] Stratified train/val/test split
- [ ] EDA plots (class counts + sample images)
- [ ] Preprocess (resize, scale)
- [ ] Extract features: pixels+PCA, color hist, HOG/LBP
- [ ] Train Logistic Regression, SVM, Random Forest via Pipelines
- [ ] Grid/Randomized search with CV
- [ ] Metrics + confusion matrices
- [ ] (Optional) Small CNN + comparison table
- [ ] Error analysis examples
- [ ] Write discussion + conclusion for the report

### Suggested library stack
- `numpy`, `matplotlib` / `seaborn`
- `scikit-learn` (core)
- `scikit-image` or OpenCV for HOG/LBP/resize (if allowed)
- `tensorflow.keras` or `torch` only for the optional small CNN

---

**Next step when ready:** implement Section 2–4 (load data + EDA + preprocess) first, then features and sklearn models.